In [163]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller
import re
import ast
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from itertools import product
import torch.utils.data as data
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV,  GridSearchCV
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor

In [117]:
tart=pd.read_csv(r'C:\Users\zions\Cocci\ValleyFever-MediaSignals\data\raw\articlewise_nlp_features.csv')

In [118]:
tart.columns

Index(['year_month', 'query_term', 'title', 'published', 'link', 'source',
       'summary', 'real_url', 'article_text', 'article_detail',
       'source classes', 'County', 'title_missing', 'body_missing',
       'title_label', 'title_positive', 'title_neutral', 'title_negative',
       'title_sentiment', 'title_score', 'body_label', 'body_positive',
       'body_neutral', 'body_negative', 'body_sentiment', 'title_risk',
       'body_risk', 'date', 'month'],
      dtype='str')

In [119]:
tart=  (tart
    .groupby('year_month')
    .agg(
        article_count=('title', 'count'),
        mean_title_risk=('title_risk', 'mean'),
        mean_body_risk=('body_risk', 'mean'),
        mean_title_sentiment=('title_sentiment', 'mean'),
        mean_body_sentiment=('body_sentiment', 'mean')
    )
    .reset_index()
)

In [120]:
tart['year_month'] = (
    pd.to_datetime(tart['year_month'])
    .dt.to_period('M')
)

In [121]:
tart.columns

Index(['year_month', 'article_count', 'mean_title_risk', 'mean_body_risk',
       'mean_title_sentiment', 'mean_body_sentiment'],
      dtype='str')

In [122]:
Counties=pd.read_csv(r'C:\Users\zions\Cocci\ValleyFever-MediaSignals\data\raw\vf_clean_normal.csv')

In [123]:
Counties.head()

,County,Year-Month,num_cases,population,rate_per_100k
0,Alameda,2001-01,0.0,1469484.0,0.000000
1,Alameda,2001-02,0.0,1469484.0,0.000000
2,Alameda,2001-03,2.0,1469484.0,0.136102
3,Alameda,2001-04,2.0,1469484.0,0.136102
4,Alameda,2001-05,0.0,1469484.0,0.000000


In [124]:
Counties['year_month']=(
    pd.to_datetime(Counties['Year-Month'])
    .dt.to_period('M')
)

In [125]:
Counties= Counties[['County','year_month','num_cases','rate_per_100k']]

In [126]:
Counties = Counties.sort_values(['County', 'year_month'])

In [ ]:
Counties.groupby(by='County')['Alameda']['rate_per_100k']

In [176]:
Counties.columns

Index(['County', 'year_month', 'num_cases', 'rate_per_100k'], dtype='str')

In [127]:
County_2001= Counties.merge(tart, on='year_month', how='inner')

In [128]:
County_2001

,County,year_month,num_cases,rate_per_100k,article_count,mean_title_risk,mean_body_risk,mean_title_sentiment,mean_body_sentiment
0,Alameda,2001-12,0.0,0.000000,1,0.000000,0.031519,-0.093663,-0.620226
1,Alameda,2004-05,0.0,0.000000,1,0.000000,0.007421,0.501221,0.591084
2,Alameda,2007-09,1.0,0.068695,1,0.000000,0.003717,0.011945,-0.007069
3,Alameda,2007-12,0.0,0.000000,1,0.090909,0.044037,-0.103591,-0.524382
4,Alameda,2009-02,4.0,0.266927,1,0.083333,0.067118,-0.078763,-0.195988
...,...,...,...,...,...,...,...,...,...
3439,Ventura,2024-08,22.0,2.655587,27,0.108151,0.048071,-0.288895,-0.312775
3440,Ventura,2024-09,22.0,2.655587,30,0.103014,0.043338,-0.188976,-0.243705
3441,Ventura,2024-10,30.0,3.621255,5,0.010000,0.013117,-0.047762,-0.218098
3442,Ventura,2024-11,45.0,5.431883,7,0.116625,0.027478,-0.053856,-0.076220


In [141]:
County_2001.isna().sum()

County                  0
year_month              0
num_cases               2
rate_per_100k           2
article_count           0
mean_title_risk         0
mean_body_risk          0
mean_title_sentiment    0
mean_body_sentiment     0
dtype: int64

In [145]:
County_2001.loc[County_2001['num_cases'].isna()]

,County,year_month,num_cases,rate_per_100k,article_count,mean_title_risk,mean_body_risk,mean_title_sentiment,mean_body_sentiment
820,Merced,2001-12,NaN,NaN,1,0.000000,0.031519,-0.093663,-0.620226
823,Merced,2007-12,NaN,NaN,1,0.090909,0.044037,-0.103591,-0.524382


In [146]:
County_2001.loc[County_2001['rate_per_100k'].isna()]

,County,year_month,num_cases,rate_per_100k,article_count,mean_title_risk,mean_body_risk,mean_title_sentiment,mean_body_sentiment
820,Merced,2001-12,NaN,NaN,1,0.000000,0.031519,-0.093663,-0.620226
823,Merced,2007-12,NaN,NaN,1,0.090909,0.044037,-0.103591,-0.524382


In [147]:
Counties.loc[Counties['year_month']=='2001-12']

,County,year_month,num_cases,rate_per_100k
11,Alameda,2001-12,0.0,0.000000
299,Contra Costa,2001-12,2.0,0.205983
587,Fresno,2001-12,2.0,0.245996
875,Kern,2001-12,88.0,13.006707
1163,Los Angeles,2001-12,7.0,0.072719
1451,Merced,2001-12,NaN,NaN
1739,Monterey,2001-12,0.0,0.000000
2027,Orange,2001-12,3.0,0.103970
2315,Riverside,2001-12,4.0,0.247428
2603,Sacramento,2001-12,1.0,0.078973


****Won't use Twitter data since there is a gap between 2001 and 2011**

In [134]:
adcols=['mean_title_sentiment', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count']

acols = {
    'articles': adcols
}

target='rate_per_100k'

horizons= [1,3,6]

In [135]:
for col in adcols:
    past_mean = County_2001[col].shift(1).expanding(min_periods=1).mean()
    County_2001[col] = County_2001[col].fillna(past_mean)

In [136]:
county_data = {
    county: group.sort_values('year_month').reset_index(drop=True)
    for county, group in County_2001.groupby('County')
}

In [137]:
county_model_data = {}

for county, df in county_data.items():
    df = df.sort_values('year_month').copy()

    df['log_rate'] = np.log1p(df[target])
    df['logdiff_rate'] = df['log_rate'].diff()

    df = df.dropna(subset=['logdiff_rate']).reset_index(drop=True)

    county_model_data[county] = {
        'df': df,
        'y': df['logdiff_rate'],
        'dates': df['year_month'],
        'X': {
    name: df[cols]
    for name, cols in acols.items()
}
}

In [182]:
print(county_model_data['Alameda'].keys())

dict_keys(['df', 'y', 'dates', 'X'])


In [187]:
alameda = county_model_data['Alameda']['df'].copy()

print(alameda.dtypes)
print(alameda[['year_month', 'logdiff_rate']].head(10))
print(type(alameda['logdiff_rate'].iloc[0]))

County                        str
year_month              period[M]
num_cases                 float64
rate_per_100k             float64
article_count               int64
mean_title_risk           float64
mean_body_risk            float64
mean_title_sentiment      float64
mean_body_sentiment       float64
log_rate                  float64
logdiff_rate              float64
dtype: object
  year_month  logdiff_rate
0    2004-05      0.000000
1    2007-09      0.066438
2    2007-12     -0.066438
3    2009-02      0.236594
4    2009-08     -0.171995
5    2009-09     -0.064599
6    2010-12      0.234521
7    2011-01     -0.002672
8    2011-02     -0.053120
9    2011-03      0.053120
<class 'numpy.float64'>


In [138]:
for county, data in county_model_data.items():

    df = data['df']
    original = county_data[county].set_index('year_month')

    for col in acols['articles']:
        missing = df[col].isna()

        df.loc[missing, col] = df.loc[missing, 'year_month'].map(
            original[col]
        )

    data['X'] = {
        name: df[cols]
        for name, cols in acols.items()
    }

In [139]:
def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train, len(X) - horizon + 1, step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_train": y.iloc[:train_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits

initial_train = 120

county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = (
                walk_forward_splits(
                    X,
                    y,
                    initial_train=initial_train,
                    horizon=horizon,
                    step=1
                )
            )

In [205]:
print(split["y_test"].index)
print(split["y_test"].head())

RangeIndex(start=157, stop=163, step=1)
157    0.273168
158    0.033578
159    0.000000
160    0.234410
161    0.330601
Name: logdiff_rate, dtype: float64


In [92]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}

In [206]:

from tqdm.auto import tqdm

XGB_results = {}
XGB_best_params = {}
XGB_importance_rows = []

# Count county / feature combinations
total_combinations = sum(
    1
    for county, feature_sets in county_splits.items()
    for feature_name, horizons in feature_sets.items()
    if horizons.get(1)
)

with tqdm(total=total_combinations, desc="XGBoost") as pbar:

    for county, feature_sets_for_county in county_splits.items():

        for feature_name, horizons_for_feature in feature_sets_for_county.items():

            pbar.set_postfix_str(f"{county} | {feature_name}")

            # Tune using the first training window
            initial_splits = horizons_for_feature[1]

            if not initial_splits:
                pbar.update(1)
                continue

            first_split = initial_splits[0]

            X_initial = first_split["X_train"]
            y_initial = first_split["y_train"].to_numpy().ravel()

            # Hyperparameter search using chronological CV
            search = RandomizedSearchCV(
                estimator=XGBRegressor(
                    objective="reg:squarederror",
                    eval_metric="rmse",
                    tree_method="hist",
                    random_state=42,
                    n_jobs=1
                ),
                param_distributions=param_distributions,
                n_iter=12,
                scoring="neg_mean_squared_error",
                cv=TimeSeriesSplit(n_splits=5),
                random_state=42,
                n_jobs=-1,
                verbose=0
            )

            search.fit(X_initial, y_initial)

            best_params = search.best_params_

            XGB_best_params[(county, feature_name)] = best_params

            # Evaluate every horizon
            for horizon, splits_for_horizon in horizons_for_feature.items():

                forecasts = []

                for i, split in enumerate(
                    tqdm(
                        splits_for_horizon,
                        desc=f"{county} | {feature_name} | H={horizon}",
                        leave=False
                    )
                ):

                    X_train = split["X_train"]
                    X_test = split["X_test"]

                    y_train = split["y_train"].to_numpy().ravel()
                    y_test = split["y_test"].to_numpy().ravel()

                    model = XGBRegressor(
                        **best_params,
                        objective="reg:squarederror",
                        eval_metric="rmse",
                        tree_method="hist",
                        random_state=42,
                        n_jobs=1
                    )

                    model.fit(X_train, y_train)

                    predictions = model.predict(X_test)

                    # Preserve original test dates
                    year_months = split["y_test"].index

                    # Store predictions
                    for lead, (year_month, actual, predicted) in enumerate(
                        zip(year_months, y_test, predictions),
                        start=1
                    ):

                        forecasts.append({
                            "year_month": year_month,
                            "split": i,
                            "lead": lead,
                            "actual": actual,
                            "predicted": predicted
                        })

                    # Store feature importance
                    for feature, importance in zip(
                        X_train.columns,
                        model.feature_importances_
                    ):

                        XGB_importance_rows.append({
                            "County": county,
                            "Feature_Set": feature_name,
                            "Horizon": horizon,
                            "Split": i,
                            "Feature": feature,
                            "Importance": importance
                        })

                # Store results
                XGB_results[
                    (county, feature_name, horizon)
                ] = pd.DataFrame(forecasts)

            pbar.update(1)

# Combine feature importance records
XGB_importance = pd.DataFrame(XGB_importance_rows)



c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
XGBoost: 100%|██████████| 21/21 [02:22<00:00,  6.77s/it, Ventura | articles]        


In [208]:
XGB_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in XGB_results.items()
    ],
    ignore_index=True
)

XGB_predictions_df.to_csv(
    "xgboost_2001_predictions.csv",
    index=False
)

pd.DataFrame(XGB_importance_rows).to_csv(
    "xgboost_feature_importance_2001.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in XGB_best_params.items()
]).to_csv(
    "xgboost_best_params_2001.csv",
    index=False
)

In [209]:
evaluation_rows = []

for (county, feature_name, horizon), df in XGB_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "XGBoost",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

XBG_evaluations_table = pd.DataFrame(evaluation_rows)

XBG_evaluations_table = XBG_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

XBG_evaluations_table.to_csv('xgboost_2001_Media_Evals.csv', index=False)

In [210]:
def select_sarimax_order(
        y_train,
        X_train,
        seasonal_period=12,
        trace=False,
):
    automatic_model = auto_arima(
        y=y_train,
        X=X_train,
        seasonal=True,
        m=seasonal_period,
        test="kpss",
        seasonal_test="ocsb",
        start_p=0,
        start_q=0,
        max_p=4,
        max_q=4,
        start_P=0,
        start_Q=0,
        max_P=2,
        max_Q=2,
        max_d=2,
        max_D=1,
        stepwise=True,
        information_criterion="aic",
        trace=trace,
    )

    trend = "c" if automatic_model.with_intercept else "n"

    return {
        "order": automatic_model.order,
        "seasonal_order": automatic_model.seasonal_order,
        "trend": trend
    }

In [212]:
def sarimax(X_train, Y_train, X_test, Y_test, params):

    model = SARIMAX(
        endog=Y_train,
        exog=X_train,
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        trend=params["trend"]
    ).fit(disp=False)

    forecast = model.get_forecast(
        steps=len(Y_test),
        exog=X_test
    )

    return {
        "model": model,
        "forecast": forecast.predicted_mean,
        "order": params["order"],
        "seasonal_order": params["seasonal_order"],
        "trend": params["trend"],
        "converged": model.mle_retvals.get("converged", None)
    }

In [213]:
import warnings
warnings.filterwarnings('ignore')

In [214]:
from tqdm.auto import tqdm

SARIMAX_results = {}
SARIMAX_best_params = {}
SARIMAX_importance_rows = []

# Calculate total combinations
total_combinations = sum(
    1
    for county, feature_sets in county_splits.items()
    for feature_name, horizons in feature_sets.items()
    for horizon, splits in horizons.items()
    if splits
)

with tqdm(total=total_combinations, desc="SARIMAX Models") as pbar:

    for county, feature_sets in county_splits.items():

        for feature_name, horizons_for_feature in feature_sets.items():

            for horizon, splits in horizons_for_feature.items():

                if not splits:
                    continue

                pbar.set_postfix_str(
                    f"{county} | {feature_name} | H={horizon}"
                )

                # Tune ONCE using first training window
                first_split = splits[0]

                params = select_sarimax_order(
                    first_split["y_train"],
                    first_split["X_train"],
                    seasonal_period=12,
                    trace=False
                )

                SARIMAX_best_params[
                    (county, feature_name, horizon)
                ] = params

                forecasts = []

                # Walk-forward progress bar
                for i, split in enumerate(
                    tqdm(
                        splits,
                        desc=f"{county} | {feature_name} | H={horizon}",
                        leave=False
                    )
                ):

                    result = sarimax(
                        X_train=split["X_train"],
                        Y_train=split["y_train"],
                        X_test=split["X_test"],
                        Y_test=split["y_test"],
                        params=params
                    )

                    predictions = result["forecast"]

                    actuals = split["y_test"].to_numpy().ravel()

                    # Preserve year_month from the test index
                    year_months = split["y_test"].index

                    for lead, (year_month, actual, predicted) in enumerate(
                        zip(year_months, actuals, predictions),
                        start=1
                    ):

                        forecasts.append({
                            "year_month": year_month,
                            "split": i,
                            "lead": lead,
                            "actual": actual,
                            "predicted": predicted,
                            "converged": result["converged"]
                        })

                    # Store coefficients
                    for feature, coefficient in zip(
                        split["X_train"].columns,
                        result["model"].params[
                            split["X_train"].columns
                        ]
                    ):

                        SARIMAX_importance_rows.append({
                            "County": county,
                            "Feature_Set": feature_name,
                            "Horizon": horizon,
                            "Split": i,
                            "Feature": feature,
                            "Coefficient": coefficient
                        })

                # Store results
                SARIMAX_results[
                    (county, feature_name, horizon)
                ] = pd.DataFrame(forecasts)

                pbar.update(1)

# Convert coefficient records into DataFrame
SARIMAX_importance = pd.DataFrame(SARIMAX_importance_rows)

SARIMAX Models: 100%|██████████| 63/63 [34:25<00:00, 32.79s/it, Ventura | articles | H=6]        


In [221]:
smax_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in XGB_results.items()
    ],
    ignore_index=True
)

smax_predictions_df.to_csv(
    "SARIMAX_2001_predictions.csv",
    index=False
)

pd.DataFrame(SARIMAX_importance_rows).to_csv(
    "SARIMAX_feature_importance_2001.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        "Horizon": horizon,
        **params
    }
    for (county, feature_name, horizon), params
    in SARIMAX_best_params.items()
]).to_csv(
    "SARIMAX_best_params_2001.csv",
    index=False
)

In [168]:
evaluation_rows = []

for (county, feature_name, horizon), df in SARIMAX_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "SARIMAX",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

SARIMAX_evaluations_table = pd.DataFrame(evaluation_rows)

SARIMAX_evaluations_table = SARIMAX_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

SARIMAX_evaluations_table.to_csv('SARIMAX_2001_Media_Evals.csv', index=False)

In [216]:
def RandForest(Xtrain, Ytrain, Xtest, Ytest):

    # Make sure the target is one-dimensional
    Ytrain = np.asarray(Ytrain).reshape(-1)
    Ytest = np.asarray(Ytest).reshape(-1)

    fr = RandomForestRegressor(random_state=5)

    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 1.0]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    grid_search = GridSearchCV(
        estimator=fr,
        param_grid=param_grid,
        cv=tscv,
        n_jobs=-1,
        scoring='neg_mean_squared_error',
        verbose=2
    )

    # Tune on training data only
    grid_search.fit(Xtrain, Ytrain)

    # Best estimator has already been fitted on Xtrain, Ytrain
    model = grid_search.best_estimator_

    # Forecast the held-out period
    fpreds = model.predict(Xtest)

    # Evaluate only against the corresponding test observations
    evaluation = Evaluator(Ytest, fpreds)

    return {
        'model': model,
        'evaluation': evaluation,
        'forecast': fpreds,
        'parameters': grid_search.best_params_,
        'cv_rmse': np.sqrt(-grid_search.best_score_)
    }

In [218]:
from tqdm.auto import tqdm

forest_results = {}
forest_best_params = {}
forest_importance_rows = []

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 1.0]
}

# Count county / feature combinations
total_combinations = sum(
    1
    for county, feature_sets in county_splits.items()
    for feature_name, horizons in feature_sets.items()
    if horizons.get(1)
)

with tqdm(total=total_combinations, desc="Random Forest") as pbar:

    for county, feature_sets_for_county in county_splits.items():

        for feature_name, horizons_for_feature in feature_sets_for_county.items():

            pbar.set_postfix_str(f"{county} | {feature_name}")

            # Tune hyperparameters using the first 120-month window
            initial_splits = horizons_for_feature[1]

            if not initial_splits:
                pbar.update(1)
                continue

            first_split = initial_splits[0]

            X_initial = first_split["X_train"]
            y_initial = first_split["y_train"].to_numpy().ravel()

            search = GridSearchCV(
                RandomForestRegressor(
                    random_state=5,
                    n_jobs=1
                ),
                param_grid=param_grid,
                cv=TimeSeriesSplit(n_splits=5),
                scoring="neg_mean_squared_error",
                n_jobs=-1,
                verbose=0
            )

            search.fit(X_initial, y_initial)

            best_params = search.best_params_

            forest_best_params[(county, feature_name)] = best_params

            # Evaluate each horizon
            for horizon, splits_for_horizon in horizons_for_feature.items():

                forecasts = []

                for i, split in enumerate(
                    tqdm(
                        splits_for_horizon,
                        desc=f"{county} | {feature_name} | H={horizon}",
                        leave=False
                    )
                ):

                    X_train = split["X_train"]
                    X_test = split["X_test"]
                    y_train = split["y_train"]
                    y_test = split["y_test"]

                    model = RandomForestRegressor(
                        **best_params,
                        random_state=5,
                        n_jobs=-1
                    )

                    model.fit(X_train, y_train)
                    predictions = model.predict(X_test)

                    actuals = y_test.to_numpy().ravel()
                    year_months = y_test.index

                    # Store predictions with original dates
                    for lead, (year_month, actual, predicted) in enumerate(
                        zip(year_months, actuals, predictions),
                        start=1
                    ):

                        forecasts.append({
                            "year_month": year_month,
                            "split": i,
                            "lead": lead,
                            "actual": actual,
                            "predicted": predicted
                        })

                    # Store feature importances
                    for feature, importance in zip(
                        X_train.columns,
                        model.feature_importances_
                    ):

                        forest_importance_rows.append({
                            "County": county,
                            "Feature_Set": feature_name,
                            "Horizon": horizon,
                            "Split": i,
                            "Feature": feature,
                            "Importance": importance
                        })

                # Store results
                forest_results[
                    (county, feature_name, horizon)
                ] = pd.DataFrame(forecasts)

            pbar.update(1)

forest_importance = pd.DataFrame(forest_importance_rows)

Random Forest:   0%|          | 0/21 [00:00<?, ?it/s, Alameda | articles]c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration

In [222]:
forest_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in forest_results.items()
    ],
    ignore_index=True
)

forest_predictions_df.to_csv(
    "RandFor_2001_predictions.csv",
    index=False
)

pd.DataFrame(forest_importance_rows).to_csv(
    "RandFor_feature_importance_2001.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in forest_best_params.items()
]).to_csv(
    "RandFor_best_params_2001.csv",
    index=False
)

In [220]:
evaluation_rows = []

for (county, feature_name, horizon), df in forest_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "RandFor",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

XBG_evaluations_table = pd.DataFrame(evaluation_rows)

XBG_evaluations_table = XBG_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

XBG_evaluations_table.to_csv('Forest_2001_Media_Evals.csv', index=False)

In [ ]:
def create_dataset(dataset, lookback):
    X, y = [], []

    for i in range(len(dataset) - lookback):
        
        # Past log_rate + exogenous variables
        X.append(
            dataset[i:i+lookback, :]
        )
        
        # Next log_rate value (column 0)
        y.append(
            dataset[i+lookback, 0]
        )

    return (
        torch.tensor(
            np.array(X),
            dtype=torch.float32
        ),
        torch.tensor(
            np.array(y),
            dtype=torch.float32
        ).reshape(-1, 1)
    )

In [223]:
class LSTMModel(nn.Module):
    def __init__(
        self,
        n_features,
        hidden_size=50,
        num_layers=1,
        dropout=0
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.linear = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.linear(x)

        return x
loss_fn = nn.MSELoss()

In [224]:
def create_sequences(X, y, lookback):
    """
    Convert time-series data into LSTM sequences.

    X shape: (observations, features)
    y shape: (observations, 1)

    Returns:
        X_seq: (samples, lookback, features)
        y_seq: (samples, 1)
    """

    X_seq = []
    y_seq = []

    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])

    return (
        torch.tensor(np.array(X_seq), dtype=torch.float32),
        torch.tensor(np.array(y_seq), dtype=torch.float32)
    )


def summarize_lstm(model):
    """
    Create a simple architecture/parameter summary for an LSTM.
    """

    rows = []

    total_params = 0

    for name, parameter in model.named_parameters():

        num_params = parameter.numel()
        total_params += num_params

        rows.append({
            "name": name,
            "shape": tuple(parameter.shape),
            "parameters": num_params
        })

    summary = pd.DataFrame(rows)

    return summary, total_params


def evaluate_lstm(
    y_true,
    y_pred,
    y_scaler=None
):
    """
    Evaluate LSTM predictions.

    If y_scaler is supplied, predictions and y_true are
    converted back to their original scale first.
    """

    if y_scaler is not None:

        y_true = y_scaler.inverse_transform(
            np.asarray(y_true).reshape(-1, 1)
        ).ravel()

        y_pred = y_scaler.inverse_transform(
            np.asarray(y_pred).reshape(-1, 1)
        ).ravel()

    else:

        y_true = np.asarray(y_true).ravel()
        y_pred = np.asarray(y_pred).ravel()

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


def select_lstm_model(
    X_train,
    y_train,
    X_test,
    y_test,
    y_scaler=None,
    lookbacks=(3, 6, 12),
    hidden_sizes=(16, 32, 64),
    num_layers_list=(1, 2),
    learning_rates=(0.001, 0.0001),
    batch_sizes=(8, 16),
    epochs=28,
    validation_fraction=0.20,
    random_state=42
):

    torch.manual_seed(random_state)
    np.random.seed(random_state)

    # --------------------------------------------------
    # Make sure y is 2D
    # --------------------------------------------------

    y_train = np.asarray(y_train).reshape(-1, 1)
    y_test = np.asarray(y_test).reshape(-1, 1)

    # --------------------------------------------------
    # Chronological train / validation split
    # --------------------------------------------------

    split = int(len(X_train) * (1 - validation_fraction))

    X_fit = X_train[:split]
    y_fit = y_train[:split]

    X_val = X_train[split:]
    y_val = y_train[split:]

    # --------------------------------------------------
    # Hyperparameter search
    # --------------------------------------------------

    search_results = []

    for lookback in lookbacks:

        # Need enough observations for both sets
        if len(X_fit) <= lookback or len(X_val) == 0:
            continue

        # Give validation the previous lookback observations
        X_val_context = np.concatenate(
            [X_fit[-lookback:], X_val],
            axis=0
        )

        y_val_context = np.concatenate(
            [y_fit[-lookback:], y_val],
            axis=0
        )

        X_fit_seq, y_fit_seq = create_sequences(
            X_fit,
            y_fit,
            lookback
        )

        X_val_seq, y_val_seq = create_sequences(
            X_val_context,
            y_val_context,
            lookback
        )

        for hidden_size in hidden_sizes:

            for num_layers in num_layers_list:

                for lr in learning_rates:

                    for batch_size in batch_sizes:

                        # ------------------------------
                        # Fresh model
                        # ------------------------------

                        model = LSTMModel(
                            n_features=X_fit_seq.shape[2],
                            hidden_size=hidden_size,
                            num_layers=num_layers
                        )

                        optimizer = torch.optim.Adam(
                            model.parameters(),
                            lr=lr
                        )

                        loss_fn = nn.MSELoss()

                        train_loader = DataLoader(
                            TensorDataset(
                                X_fit_seq,
                                y_fit_seq
                            ),
                            batch_size=batch_size,
                            shuffle=True
                        )

                        # ------------------------------
                        # Train
                        # ------------------------------

                        for epoch in range(epochs):

                            model.train()

                            for X_batch, y_batch in train_loader:

                                optimizer.zero_grad()

                                predictions = model(X_batch)

                                loss = loss_fn(
                                    predictions,
                                    y_batch
                                )

                                loss.backward()
                                optimizer.step()

                        # ------------------------------
                        # Validation
                        # ------------------------------

                        model.eval()

                        with torch.no_grad():

                            val_predictions = model(
                                X_val_seq
                            )

                            val_loss = loss_fn(
                                val_predictions,
                                y_val_seq
                            ).item()

                        search_results.append({
                            "lookback": lookback,
                            "hidden_size": hidden_size,
                            "num_layers": num_layers,
                            "learning_rate": lr,
                            "batch_size": batch_size,
                            "validation_loss": val_loss
                        })

    # --------------------------------------------------
    # Find best parameters
    # --------------------------------------------------

    search_results = pd.DataFrame(search_results)

    best_row = (
        search_results
        .sort_values("validation_loss")
        .iloc[0]
    )

    best_params = {
        "lookback": int(best_row["lookback"]),
        "hidden_size": int(best_row["hidden_size"]),
        "num_layers": int(best_row["num_layers"]),
        "learning_rate": float(best_row["learning_rate"]),
        "batch_size": int(best_row["batch_size"]),
        "epochs": epochs
    }

    # --------------------------------------------------
    # Retrain final model using ALL training data
    # --------------------------------------------------

    lookback = best_params["lookback"]

    X_train_seq, y_train_seq = create_sequences(
        X_train,
        y_train,
        lookback
    )

    X_test_context = np.concatenate(
        [X_train[-lookback:], X_test],
        axis=0
    )

    y_test_context = np.concatenate(
        [y_train[-lookback:], y_test],
        axis=0
    )

    X_test_seq, y_test_seq = create_sequences(
        X_test_context,
        y_test_context,
        lookback
    )

    final_model = LSTMModel(
        n_features=X_train_seq.shape[2],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"]
    )

    optimizer = torch.optim.Adam(
        final_model.parameters(),
        lr=best_params["learning_rate"]
    )

    loss_fn = nn.MSELoss()

    train_loader = DataLoader(
        TensorDataset(
            X_train_seq,
            y_train_seq
        ),
        batch_size=best_params["batch_size"],
        shuffle=True
    )

    # --------------------------------------------------
    # Final training
    # --------------------------------------------------

    for epoch in range(epochs):

        final_model.train()

        for X_batch, y_batch in train_loader:

            optimizer.zero_grad()

            predictions = final_model(X_batch)

            loss = loss_fn(
                predictions,
                y_batch
            )

            loss.backward()
            optimizer.step()

    # --------------------------------------------------
    # Test predictions
    # --------------------------------------------------

    final_model.eval()

    with torch.no_grad():

        test_predictions = final_model(
            X_test_seq
        ).numpy()

    # --------------------------------------------------
    # Evaluation
    # --------------------------------------------------

    evaluation = evaluate_lstm(
        y_test_seq.numpy(),
        test_predictions,
        y_scaler=y_scaler
    )

    # --------------------------------------------------
    # Model summary
    # --------------------------------------------------

    model_summary, total_params = summarize_lstm(
        final_model
    )

    return {
        "model": final_model,
        "best_params": best_params,
        "evaluation": evaluation,
        "model_summary": model_summary,
        "total_parameters": total_params,
        "search_results": search_results,
        "predictions": test_predictions,
        "y_test": y_test_seq.numpy()
    }

In [225]:
def fit_lstm_fixed_params(X_train, y_train, X_test, y_test,
                          params, random_state=42):
    """Scale, train a fresh LSTM, and predict one walk-forward split."""

    # Fit scalers on training data only
    X_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_scaled = X_scaler.fit_transform(X_train)
    X_test_scaled = X_scaler.transform(X_test)

    y_train_scaled = y_scaler.fit_transform(
        np.asarray(y_train).reshape(-1, 1)
    )
    y_test_scaled = y_scaler.transform(
        np.asarray(y_test).reshape(-1, 1)
    )

    lookback = params["lookback"]

    if len(X_train_scaled) <= lookback:
        raise ValueError("Not enough training observations for lookback.")

    # Training sequences
    X_train_seq, y_train_seq = create_sequences(
        X_train_scaled, y_train_scaled, lookback
    )

    # Include preceding training observations as test context
    X_test_context = np.concatenate(
        [X_train_scaled[-lookback:], X_test_scaled], axis=0
    )
    y_test_context = np.concatenate(
        [y_train_scaled[-lookback:], y_test_scaled], axis=0
    )

    X_test_seq, _ = create_sequences(
        X_test_context, y_test_context, lookback
    )

    # Reproducible model initialization
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    model = LSTMModel(
        n_features=X_train_seq.shape[2],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"]
    )

    optimizer = torch.optim.Adam(
        model.parameters(), lr=params["learning_rate"]
    )
    loss_fn = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(X_train_seq, y_train_seq),
        batch_size=params["batch_size"],
        shuffle=True
    )

    for epoch in range(params["epochs"]):
        model.train()

        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_scaled = model(X_test_seq).cpu().numpy()

    # Return predictions on the original log-difference scale
    predictions = y_scaler.inverse_transform(pred_scaled).ravel()
    actual = np.asarray(y_test).reshape(-1)

    return model, actual, predictions

In [226]:
from tqdm.auto import tqdm

LSTM_results = {}
LSTM_best_params = {}

# Count county / feature combinations
total_combinations = sum(
    1
    for county, feature_sets in county_splits.items()
    for feature_name, horizons in feature_sets.items()
    if horizons.get(1)
)

with tqdm(total=total_combinations, desc="LSTM Models") as pbar:

    for county, feature_sets_for_county in county_splits.items():

        for feature_name, horizons_for_feature in feature_sets_for_county.items():

            pbar.set_postfix_str(f"{county} | {feature_name}")

            # ------------------------------------------
            # Tune using initial training window
            # ------------------------------------------

            initial_splits = horizons_for_feature[1]

            if not initial_splits:
                pbar.update(1)
                continue

            first_split = initial_splits[0]

            X_initial = first_split["X_train"]
            y_initial = first_split["y_train"]

            # Scale initial training data
            X_scaler = StandardScaler()
            y_scaler = StandardScaler()

            X_initial_scaled = X_scaler.fit_transform(X_initial)

            y_initial_scaled = y_scaler.fit_transform(
                np.asarray(y_initial).reshape(-1, 1)
            )

            X_initial_test = X_scaler.transform(
                first_split["X_test"]
            )

            y_initial_test = y_scaler.transform(
                np.asarray(first_split["y_test"]).reshape(-1, 1)
            )

            # ------------------------------------------
            # Hyperparameter selection (once)
            # ------------------------------------------

            selection = select_lstm_model(
                X_initial_scaled,
                y_initial_scaled,
                X_initial_test,
                y_initial_test,
                y_scaler=y_scaler,
                lookbacks=(3, 6, 12),
                hidden_sizes=(16, 32, 64),
                num_layers_list=(1, 2),
                learning_rates=(0.001, 0.0001),
                batch_sizes=(8, 16),
                epochs=28,
                validation_fraction=0.20,
                random_state=42
            )

            best_params = selection["best_params"]

            LSTM_best_params[(county, feature_name)] = best_params

            # ------------------------------------------
            # Evaluate every forecast horizon
            # ------------------------------------------

            for horizon, splits_for_horizon in horizons_for_feature.items():

                forecasts = []

                for i, split in enumerate(
                    tqdm(
                        splits_for_horizon,
                        desc=f"{county} | {feature_name} | H={horizon}",
                        leave=False
                    )
                ):

                    # Extract existing walk-forward split
                    X_train = split["X_train"]
                    X_test = split["X_test"]

                    y_train = split["y_train"]
                    y_test = split["y_test"]

                    # Fit fresh LSTM using selected parameters
                    model, actual, predictions = fit_lstm_fixed_params(
                        X_train.to_numpy(),
                        y_train.to_numpy(),
                        X_test.to_numpy(),
                        y_test.to_numpy(),
                        params=best_params,
                        random_state=42 + i
                    )

                    # Preserve original test dates
                    year_months = y_test.index

                    # Store predictions
                    for lead, (year_month, a, p) in enumerate(
                        zip(year_months, actual, predictions),
                        start=1
                    ):

                        forecasts.append({
                            "year_month": year_month,
                            "split": i,
                            "lead": lead,
                            "actual": a,
                            "predicted": p
                        })

                # Store results
                LSTM_results[
                    (county, feature_name, horizon)
                ] = pd.DataFrame(forecasts)

            pbar.update(1)

LSTM Models:   0%|          | 0/21 [00:00<?, ?it/s, Alameda | articles]c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
LSTM Models:   5%|▍         | 1/21 [03:17<1:05:48, 197.44s/it, Contra Costa | articles]c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
LSTM Models:  10%|▉         | 2/21 [06:46<1:04:43, 204.42s/it, Fresno | articles]      c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
LSTM Models:  14%|█▍        | 3/21 [09:54<59

In [227]:
LSTM_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in LSTM_results.items()
    ],
    ignore_index=True
)

LSTM_predictions_df.to_csv(
    "LSTM_2001_predictions.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in LSTM_best_params.items()
]).to_csv(
    "LSTM_best_params_2001.csv",
    index=False
)

In [228]:
evaluation_rows = []

for (county, feature_name, horizon), df in LSTM_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "RandFor",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

XBG_evaluations_table = pd.DataFrame(evaluation_rows)

XBG_evaluations_table = XBG_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

XBG_evaluations_table.to_csv('LSTM_2001_Media_Evals.csv', index=False)